# General Tool-Using Agent With Groq And Gradio

This notebook builds a general agent.

It is not only a restaurant agent.

It can answer normal questions using the LLM, and it can use tools when the model needs current information.

Tools included:

- current date and time tool
- weather tool using Open-Meteo, free and no API key
- web search tool using DuckDuckGo HTML search, free and no API key
- Gradio app

## Easy Idea

An LLM is smart, but it may not know live information.

For example, it may not know:

- today's date
- current weather
- today's news
- latest updates after its training date

So we give the agent tools.

Daily life example: a teacher may know many things, but for today's weather the teacher checks a weather app. The weather app is a tool.

## Free Tool Suggestions

| Need | Free-friendly tool |
| --- | --- |
| Current date/time | Python `datetime` |
| Weather | Open-Meteo API, no key |
| Web search | DuckDuckGo search, no key |
| LLM | Groq API key |

Important: free tools can still have limits or temporary blocking if used too much.

## Step 1: Imports

We import Python tools, LangChain tools, LangGraph, Groq, and Gradio.

In [7]:
import json
import os
import re
import urllib.parse
import urllib.request
from datetime import datetime
from html import unescape
from zoneinfo import ZoneInfo

import gradio as gr
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import START, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

/var/folders/gg/_n7zf4557y32204w7jgjm3rw0000gn/T/ipykernel_71488/3752263367.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


## Step 2: Load Groq Key

Add your Groq key in `.env`:

```env
GROQ_API_KEY=your_key_here
```

Do not upload `.env` to GitHub.

In [8]:
load_dotenv()

print("GROQ_API_KEY:", "set" if os.getenv("GROQ_API_KEY") else "missing")

GROQ_API_KEY: set


## Step 3: Create Tools

A tool is a Python function that the LLM can call.

The LLM does not directly know live weather or live search results.

When needed, it calls these tools and then uses their results to answer.

In [ ]:
@tool
def get_current_datetime(timezone: str = "Asia/Karachi") -> str:
    """Get the current date and time for a timezone. Use for questions about today, date, time, or now."""
    try:
        now = datetime.now(ZoneInfo(timezone))
    except Exception:
        now = datetime.now()

    return now.strftime("%A, %B %d, %Y %I:%M %p")


@tool
def get_weather(city: str) -> str:
    """Get current weather for a city using Open-Meteo. Use for weather questions."""
    city = city.strip()
    if not city:
        return "Please provide a city name."

    geocode_url = "https://geocoding-api.open-meteo.com/v1/search?" + urllib.parse.urlencode({
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    })

    with urllib.request.urlopen(geocode_url, timeout=15) as response:
        geocode_data = json.loads(response.read().decode("utf-8"))

    results = geocode_data.get("results", [])
    if not results:
        return f"I could not find weather coordinates for {city}."

    place = results[0]
    latitude = place["latitude"]
    longitude = place["longitude"]
    place_name = place.get("name", city)
    country = place.get("country", "")

    weather_url = "https://api.open-meteo.com/v1/forecast?" + urllib.parse.urlencode({
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "timezone": "auto"
    })

    with urllib.request.urlopen(weather_url, timeout=15) as response:
        weather_data = json.loads(response.read().decode("utf-8"))

    current = weather_data.get("current", {})
    temperature = current.get("temperature_2m")
    humidity = current.get("relative_humidity_2m")
    wind = current.get("wind_speed_10m")

    return f"Current weather in {place_name}, {country}: temperature {temperature} C, humidity {humidity}%, wind speed {wind} km/h."


@tool
def web_search(query: str) -> str:
    """Search the web for current or latest information. Use for latest news, recent events, or facts that may have changed."""
    query = query.strip()
    if not query:
        return "Please provide a search query."
    url = "https://duckduckgo.com/html/?" + urllib.parse.urlencode({"q": query})
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

    with urllib.request.urlopen(request, timeout=20) as response:
        html = response.read().decode("utf-8", errors="ignore")

    results = []
    blocks = re.findall(r'<a rel="nofollow" class="result__a" href="(.*?)">(.*?)</a>', html)

    for link, title_html in blocks[:5]:
        title = re.sub("<.*?>", "", title_html)
        title = unescape(title).strip()
        link = unescape(link).strip()
        results.append(f"- {title}: {link}")

    if not results:
        return "No search results found. Try a different query."

    return "Search results:\n" + "\n".join(results)

In [14]:
import duckduckgo_search
@tool
def web_search(query: str) -> str:
    """Search the web for current or latest information."""
    return duckduckgo_search.invoke(query)

## Step 4: Test Tools Directly

Before connecting tools to the LLM, test them like normal Python functions.

In [15]:
print(get_current_datetime.invoke({"timezone": "Asia/Karachi"}))

Monday, July 06, 2026 09:55 PM


In [11]:
print(get_weather.invoke({"city": "Karachi"}))

Current weather in Karachi, Pakistan: temperature 29.8 C, humidity 81%, wind speed 9.2 km/h.


## Step 5: Create LLM And Bind Tools

`bind_tools` means we give the LLM permission to call these tools.

The LLM will decide when to call a tool.

Example:

- If user asks today's date, call date tool.
- If user asks weather, call weather tool.
- If user asks latest news, call search tool.
- If user asks a normal concept question, answer directly.

In [7]:
tools = [get_current_datetime, get_weather, web_search]

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
llm_with_tools = llm.bind_tools(tools)

## Step 6: Build LangGraph Tool Agent

This graph has two main parts:

1. `assistant`: the LLM decides what to do
2. `tools`: LangGraph runs the selected tool

If the LLM calls a tool, the graph goes to the tool node and then back to the assistant.

If no tool is needed, the graph ends.

In [8]:
system_prompt = """
You are a helpful general assistant.
Use tools when the user asks for current date, current time, weather, latest news, recent events, or live/current information.
If the answer does not need current information, answer directly.
When using web search, summarize the result and mention that it came from search results.
Keep answers simple and beginner friendly.
"""


def assistant_node(state: MessagesState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


tool_node = ToolNode(tools)

tool_agent_builder = StateGraph(MessagesState)
tool_agent_builder.add_node("assistant", assistant_node)
tool_agent_builder.add_node("tools", tool_node)

tool_agent_builder.add_edge(START, "assistant")
tool_agent_builder.add_conditional_edges("assistant", tools_condition)
tool_agent_builder.add_edge("tools", "assistant")

tool_agent = tool_agent_builder.compile()

## Step 7: Test The Agent

Try questions that need tools and questions that do not.

In [9]:
response = tool_agent.invoke({"messages": [HumanMessage(content="What is today's date in Pakistan?")]})
response["messages"][-1].content

"Note: The time is based on the default timezone 'Asia/Karachi' which is used in the function call."

In [10]:
response = tool_agent.invoke({"messages": [HumanMessage(content="What is the current weather in Karachi?")]})
response["messages"][-1].content

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_weather>{"city": "Karachi"}'}}

In [13]:
response = tool_agent.invoke({"messages": [HumanMessage(content="Search latest AI news today and summarize it.")]})
response["messages"][-1].content

'I was unable to find any information on the latest AI news.'

## Step 8: Gradio App

Now we make a simple app.

Type any question.

The agent will decide if it should answer directly or use a tool.

In [11]:
def general_agent_app(user_message: str) -> str:
    if not user_message.strip():
        return "Please type a question."

    try:
        response = tool_agent.invoke({"messages": [HumanMessage(content=user_message)]})
        return response["messages"][-1].content
    except Exception as error:
        return f"Error: {error}"

In [12]:
demo = gr.Interface(
    fn=general_agent_app,
    inputs=gr.Textbox(label="Ask anything", placeholder="Example: What is the weather in Karachi today?"),
    outputs=gr.Textbox(label="Agent Answer"),
    title="General Tool-Using Agent",
    description="Groq + LangGraph + tools for date, weather, and search."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## What You Built

You built a general agent.

The LLM is the brain.

The tools are helpers.

LangGraph is the manager that moves between the brain and the tools.

Gradio is the user interface.

This is the basic pattern for many real agent apps.